# Text-to-SQL AI Agent using OpenAI and MySQL

This project converts natural language questions into SQL queries using OpenAI API and executes them on MySQL database.

In [ ]:
!pip install openai langchain-community pymysql

In [2]:
from langchain_community.utilities import SQLDatabase


In [ ]:
host = "localhost"
port = "3306"
username = "YOUR_NAME"
password = "YOUR_PASSWORD"
database_schema = "text_to_sql"

mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"

In [4]:
db = SQLDatabase.from_uri(mysql_uri)

In [5]:
print(db.get_usable_table_names())


['2017_budgets', 'customers', 'products', 'regions', 'sales_order', 'state_regions']


In [6]:
from openai import OpenAI

In [7]:
!pip install openai -q

In [8]:
!pip install --upgrade openai

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY)",
    base_url="https://openrouter.ai/api/v1"
)

response = client.chat.completions.create(
    model="openai/gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [19]:
question = "How many products are there?"

response = client.chat.completions.create(
    model="openai/gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": """
You are an AI assistant that converts natural language questions into SQL queries.

Database tables:
customers
products
regions
sales_order
state_regions
2017_budgets
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]
)

print(response.choices[0].message.content)

```sql
SELECT COUNT(*) AS total_products
FROM products;
```


In [ ]:
from langchain_community.utilities import SQLDatabase

host = "localhost"
port = "3306"
username = "YOUR_USERNAME"
password = "YOUR_PASSWORD"
database_schema = "text_to_sql"

mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"

db = SQLDatabase.from_uri(mysql_uri)

print(db.get_usable_table_names())

['2017_budgets', 'customers', 'products', 'regions', 'sales_order', 'state_regions']


In [20]:
question = "How many products are there?"

response = client.chat.completions.create(
    model="openai/gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": """
You are an AI assistant that converts natural language into SQL queries.

Database tables:
customers
products
regions
sales_order
state_regions
2017_budgets

Only give SQL query.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]
)

sql_query = response.choices[0].message.content

print("Generated SQL Query:")
print(sql_query)

print("\nDatabase Result:")
result = db.run(sql_query)

print(result)

Generated SQL Query:
SELECT COUNT(*) FROM products;

Database Result:
[(30,)]


In [23]:
questions = [
    "Show first 5 customers",
    "Show first 5 products",
    "Show first 5 orders",
    "How many products are there?"
]

for question in questions:

    print("\nQuestion:", question)

    # Generate SQL query
    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": """
You are an AI assistant that converts natural language into SQL queries.

Database tables:
customers
products
regions
sales_order
state_regions
2017_budgets

Only give SQL query.
"""
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    # Extract SQL query
    sql_query = response.choices[0].message.content

    # Remove markdown formatting
    sql_query = sql_query.replace("```sql", "")
    sql_query = sql_query.replace("```", "")
    sql_query = sql_query.strip()

    # Print generated SQL
    print("\nGenerated SQL Query:")
    print(sql_query)

    # Execute query
    print("\nDatabase Result:")

    result = db.run(sql_query)

    # Print limited output
    print(str(result)[:1000])

    print("\n--------------------------")


Question: Show first 5 customers

Generated SQL Query:
SELECT * FROM customers LIMIT 5;

Database Result:
[(1, 'Geiss Company'), (2, 'Jaxbean Group'), (3, 'Ascend Ltd'), (4, 'Eire Corp'), (5, 'Blogtags Ltd')]

--------------------------

Question: Show first 5 products

Generated SQL Query:
SELECT * 
FROM products
LIMIT 5;

Database Result:
[(1, 'Product 1'), (2, 'Product 2'), (3, 'Product 3'), (4, 'Product 4'), (5, 'Product 5')]

--------------------------

Question: Show first 5 orders

Generated SQL Query:
SELECT *
FROM sales_order
LIMIT 5;

Database Result:
[('SO - 000225', '2021-01-01', 126, 'Wholesale', 'USD', 'AXW291', 364, 27, 6, 2499.1, 14994.6, 1824.343), ('SO - 0003378', '2021-01-01', 96, 'Distributor', 'USD', 'AXW291', 488, 20, 11, 2351.7000000000003, 25868.700000000004, 1269.918), ('SO - 0005126', '2021-01-01', 8, 'Wholesale', 'USD', 'AXW291', 155, 26, 6, 978.2, 5869.200000000001, 684.74), ('SO - 0005614', '2021-01-01', 42, 'Export', 'USD', 'AXW291', 473, 7, 7, 2338.3, 16